## Import the CIFAR-10


In [ ]:
# libs needed
import tarfile
import pickle

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

print("ok")

## Load the data


In [ ]:
with tarfile.open("cifar-10-python.tar.gz", "r:gz") as tar:
    tar.extractall()

# from https://www.cs.toronto.edu/~kriz/cifar.html
# Using the unpickle function, each of the bach files contains a dictionary with the following contents:
#   data --> a 10000x3072 numpy array of uint8's.
#       - Each row of the array stores a 32x32 colour image.
#       - Where the first 1024 entries contain the red channel, then 1024 for green, and then 1024 for blue
#       - The image is stored in row-major order, so the first 32 entries of the array are red channel values of the first row of the image
# SÅ data[i] indeholder 3 arrays af 1024 hver --> [ R (1024 values), G (1024 values), B (1024 values) ]
#

#   labels --> a list of 10000 numbers in the range 0-9
#       - The number at index i indicates the label of the i'th image in the array data

def unpickle(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

In [ ]:
x_train = [] # the data
y_train = [] # the labels

for i in range(1,6):
  dict_train = unpickle(f'cifar-10-batches-py/data_batch_{i}')
  x_train.append(dict_train[b'data'])
  y_train.append(dict_train[b'labels'])

# Make them a long list instead of a list of lists
x_train = np.concatenate(x_train)
y_train = np.concatenate(y_train)

# The test data
dict_test = unpickle(f'cifar-10-batches-py/test_batch')
x_test = dict_test[b'data']
y_test = dict_test[b'labels']

# reshape so we get 50000,3,32,32 --> https://github.com/n-kostadinov/cnn-image-classification-cifar-10-from-scratch/blob/master/cnn-image-classification-cifar-10-from-scratch.ipynb

#features = batch['data'].reshape((len(batch['data']), 3, 32, 32)).transpose(0, 2, 3, 1)
x_train = x_train.reshape(-1, 3, 32, 32).astype(np.float32) / 255.0   # (N, C, H, W)
# x_train = x_train.transpose(0, 2, 3, 1)    # (N, H, W, C)

x_test  = x_test.reshape(-1, 3, 32, 32).astype(np.float32) / 255.0
# x_test = x_test.transpose(0, 2, 3, 1)


# total size
n_train = x_train.shape[0]
n_test = x_test.shape[0]
n_total = n_train + n_test





# ------------------------------------------------------------
# pick one example of each digit
# ------------------------------------------------------------
examples = []

for digit in range(10):
    idx = np.where(y_train == digit)[0][0]
    img = np.transpose(x_train[idx], (1, 2, 0))   # CHW -> HWC only for plotting
    examples.append(img)


# ------------------------------------------------------------
# create figure
# ------------------------------------------------------------
fig = plt.figure(figsize=(12,5))

gs = fig.add_gridspec(2, 10, height_ratios=[1,1.2])

# title
fig.suptitle("CIFAR-10 Dataset Images", fontsize=15)

classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
# ------------------------------------------------------------
# row of example digits
# ------------------------------------------------------------
for i in range(10):
    ax = fig.add_subplot(gs[0,i])

    ax.imshow(examples[i])
    # ax.set_title(str(i), fontsize=16)
    ax.set_title(str(classes[i]), fontsize=16)
    ax.axis("off")


## Convert to torch tensors

In [ ]:
X = torch.tensor(x_train, dtype=torch.float32)
Y = torch.tensor(y_train, dtype=torch.long)

X_test = torch.tensor(x_test, dtype=torch.float32)
Y_test = torch.tensor(y_test, dtype=torch.long)

# Setting up MobileNetV1

In [ ]:
import torch
# all nn libraries nn.layer, convs and loss functions

import torch.nn as nn
# Display Image

from IPython.display import Image

# visualisation
# !pip install torchview transformers
import torchvision
from torchview import draw_graph

# !pip install transformers
from transformers import MobileNetV1Config, MobileNetV1Model



########################################################3


class DepthWiseSeperable(nn.Module):

    def __init__(self, in_channels , out_channels , stride ):
        """
        DepthWiseSeperable block of MobileNet which performs the following operations:
        (a) depthwise convolution by applying a separate filter for each channel
        (b) pointwise convolutions are applied which combine the filtered result by implementing 1 × 1 convolution
        
            Note:
                1. groups = in_channels used for depthwise convolution
                2. in_channels and out_channels are same for depthwise convolution
                3. bias = False due to the usage of BatchNorm 
                4. To generate same height and width of output feature map as the input feature map, following should be padding for
                    * 1x1 conv : p=0
                    * 3x3 conv : p=1
                    * 5x5 conv : p=2


        Args:
          in_channels (int) : number of input channels
          out_channels (int) : number of output channels 
          stride (int) : stride used for depthwise convolution

        Attributes:
            Depthwise seperable convolutional block

        """

        super(DepthWiseSeperable,self).__init__()
        
        # groups used here
        self.depthwise = nn.Conv2d(in_channels = in_channels , out_channels = in_channels , stride = stride , padding = 1, kernel_size = 3 , groups=in_channels , bias = False)
        self.bn1 = nn.BatchNorm2d(in_channels)

        self.pointwise = nn.Conv2d(in_channels = in_channels , out_channels = out_channels , stride = 1 , padding = 0, kernel_size = 1, bias = False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU()

    def forward(self,x):

        x = self.depthwise(x)
        x = self.bn1(x)
        x = self.relu(x)
        
        x = self.pointwise(x)
        x = self.bn2(x)
        x = self.relu(x)
        
        return x



################################################################################################33


class MobileNetV1(nn.Module):
    
    def __init__(self, num_classes=10):
        
        super(MobileNetV1, self).__init__()

        # Initial convolution layer
        self.features1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias = False),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(32),
        )
        
        # Depthwise separable convolutions
        self.features = nn.Sequential(
            self.features1,                      # Inserting the above features here
            DepthWiseSeperable(32, 64, 1),
            DepthWiseSeperable(64, 128, 2),
            DepthWiseSeperable(128, 128, 1),
            DepthWiseSeperable(128, 256, 2),
            DepthWiseSeperable(256, 256, 1),
            DepthWiseSeperable(256, 512, 2),
            
            DepthWiseSeperable(512, 512, 1),
            DepthWiseSeperable(512, 512, 1),
            DepthWiseSeperable(512, 512, 1),
            DepthWiseSeperable(512, 512, 1),
            DepthWiseSeperable(512, 512, 1),

            DepthWiseSeperable(512, 1024, 2),
            DepthWiseSeperable(1024, 1024, 1)

        )
        
        # Average pooling and classifier
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# Training the model

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split
import torch.optim as optim
import torch.nn as nn
import copy

# set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

# Create datasets
full_train_dataset = TensorDataset(X, Y)
test_dataset = TensorDataset(X_test, Y_test)


#Split the fll traning dataset into train and validation
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split( 
    full_train_dataset, [train_size, val_size] )

#Create data loaders
batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True)

val_loader = DataLoader( 
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False )

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False)



# Instantiate the model
model = MobileNetV1(num_classes=10).to(device)


# Define loss function and optimizer
lossFunction = nn.CrossEntropyLoss()
optimizer = optim.Adam(params=model.parameters(), lr=0.001)


# Number of steps for training
num_epochs = 50

# # Early stopping settings
# patience = 10
# best_val_loss = float("inf")
# epochs_without_improvement = 0
# best_model_weights = copy.deepcopy(model.state_dict())


# For storing results
train_losses = []
train_accuracies = []
val_losses = [] 
val_accuracies = []


def evaluate(model, dataloader):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, Y_batch in dataloader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            outputs = model(X_batch)
            loss = lossFunction(outputs, Y_batch)

            total_loss += loss.item() * X_batch.size(0)

            predictions = torch.argmax(outputs, dim=1)
            correct += (predictions == Y_batch).sum().item()
            total += Y_batch.size(0)

    average_loss = total_loss / total
    accuracy = correct / total * 100

    return average_loss, accuracy


# Training loop
for epoch in range(num_epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for X_batch, Y_batch in train_loader:
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        # Forward pass
        outputs = model(X_batch)
        loss = lossFunction(outputs, Y_batch)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Save statistics
        running_loss += loss.item() * X_batch.size(0)

        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == Y_batch).sum().item()
        total += Y_batch.size(0)

    train_loss = running_loss / total
    train_acc = correct / total * 100

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)


    #Evaluate at epoch level
    val_loss, val_acc = evaluate(model, val_loader)

    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    


    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f}, "
        f"Train Acc: {train_acc:.2f}  |  %"
        f"Test Loss: {val_loss:.4f}, " 
        f"Test Acc: {val_acc:.2f}%"
    )


    # # Early stopping check
    # if val_loss < best_val_loss:
    #     best_val_loss = val_loss
    #     epochs_without_improvement = 0
    #     best_model_weights = copy.deepcopy(model.state_dict())
    #     # print("Validation loss improved. Saving model.")
    # else:
    #     epochs_without_improvement += 1
    #     print(f"No improvement for {epochs_without_improvement} epoch(s).")

    # if epochs_without_improvement >= patience:
    #     print("Early stopping triggered.")
    #     break




# # Restore best model weights
# model.load_state_dict(best_model_weights)


print('Training finished.')

# Evaluate !!!
print('EVAL!!!!!!!!!!!!!!!')
test_loss, test_acc = evaluate(model, test_loader)

print(f"Final Test Loss: {test_loss:.4f}")
print(f"Final Test Accuracy: {test_acc:.2f}%")

# Plot...

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Training loss")
plt.plot(val_losses, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss during training")
plt.legend()
plt.grid()
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(train_accuracies, label="Training accuracy")
plt.plot(val_accuracies, label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy during training")
plt.legend()
plt.grid()
plt.show()

# Evaluate the model

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
import torch

model.eval()

def get_predictions(model, X, Y, batch_size=256):
    all_preds = []
    all_true = []

    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            X_batch = X[i:i+batch_size]
            Y_batch = Y[i:i+batch_size]

            outputs = model(X_batch)              # logits
            preds = torch.argmax(outputs, dim=1)  # predicted class index

            all_preds.append(preds.cpu())
            all_true.append(Y_batch.cpu())

    y_pred = torch.cat(all_preds)
    y_true = torch.cat(all_true)
    return y_true, y_pred


# Get predictions
y_true_train, y_pred_train = get_predictions(model, X, Y)
y_true_test, y_pred_test = get_predictions(model, X_test, Y_test)

# Accuracy
train_acc = (y_pred_train == y_true_train).float().mean().item() * 100
test_acc = (y_pred_test == y_true_test).float().mean().item() * 100

print(f"Accuracy on training data: {train_acc:.2f}%")
print(f"Accuracy on test data: {test_acc:.2f}%")

# Class names
classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

# Classification report
print("\nClassification report on test data:")
print(classification_report(y_true_test.numpy(), y_pred_test.numpy(), target_names=classes))

## Visualize the confusion matrix


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true_test.numpy(), y_pred_test.numpy())

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
fig, ax = plt.subplots(figsize=(10, 10))
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues")
plt.title("Confusion Matrix - Test Data")
plt.show()